### **Cleaning Report Dataset Notebook:** 
This notebook prepares the raw ZüriWieNeu report data for later spatial analysis. The main objective is to inspect, clean, and structure the original CSV file so that it can be used reliably in the following notebooks. This is an important first step because the later spatial analysis depends on this preparation.

First, the raw ZüriWieNeu CSV file is loaded and inspected. The dataset contains individual infrastructure reports from the city of Zurich, including information such as the request id, submission date, update date, coordinates, service code (category), status, title, and detailed description and further information. The metadata are used to understand the meaning of the most important columns and to decide which attributes are relevant for the project.

Second, the dataset is cleaned by checking data types, missing values and unnecessary columns. The service_request_id column is used as the stable unique identifier for each report, while objectid is not used for the analysis because it is only a system identifier. The date columns are converted into datetime format, and the coordinate columns "e" and "n" are kept because they represent the report locations in the Swiss coordinate reference system EPSG:2056, because they are in meters.

Third, additional time variables are created from the requested_datetime column. These include year_requested, month_requested, and progressing_time_days, which will later make it possible to analyse how the number of reports changes over time. Optional fields such as detail is kept only as descriptive information and are not required for the main spatial analysis.

By the end of this notebook, the raw ZüriWieNeu CSV file has been transformed into a cleaner and more structured dataset and safed in the dataproccessed folder as a new CSV.

Expected output:
- a cleaned ZüriWieNeu report dataset
- stable report IDs based on service_request_id
- valid coordinate columns e and n in metres
- additional time columns for later temporal analysis
- a processed CSV file that can be used in the spatialjoin notebook

---
Firstly the necessary libary for this notebook were imported. Pandas is needed to import and clean the CSV Dataset of the Reports.

Some options were set, so that the export in the end is not influenced by to big tables.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 6)
pd.set_option("display.width", 100)
pd.set_option("display.max_colwidth", 30)

The reports CSV is read into the variable "reports". It is in a relative path (..), so the reproducibility is given. To get a clear insight on the dataset, the .head() function was used, so i can look at every data column and some examples what is written inside.

In [2]:
reports = pd.read_csv("../data/rawdata/data/zueriwieneu_data.csv")
#looking at the data first, but only two row, so it does not much space
reports.head(2)

,objectid,service_request_id,requested_datetime,...,description,url,geometry
0,1,1,2013-03-14T15:16:15,...,Auf dem Asp: Auf dem Aspha...,https://www.zueriwieneu.ch...,POINT (2678968 1247548)
1,2,2,2013-03-14T15:17:57,...,Vermessungs: Vermessungspu...,https://www.zueriwieneu.ch...,POINT (2680746 1249916)


We can see, that there is two id columns(objectid and service_request_id). The columns service_code and service_name are also idenitcal in the first two rows. There are some NaN values in the first two rows in media_url, however thats mostly not the most important column for my analsysis. There is also point geometry already included, however the reports dataset is not a geogrpahical dataset yet. For later conversion in a geo data frame, there are the given cooridnates easting(e) and northing(n) in meters (CH1903+) for conversion. (The meta data provides useful information). 

However a general overview of the data in the notebook is always useful, thats why i use the function .info(), so a get an overview on how many entries there are in total und what datatypes the columns are.

In [3]:
reports.info()

<class 'pandas.DataFrame'>
RangeIndex: 72606 entries, 0 to 72605
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   objectid              72606 non-null  int64
 1   service_request_id    72606 non-null  int64
 2   requested_datetime    72606 non-null  str  
 3   agency_sent_datetime  71785 non-null  str  
 4   updated_datetime      72606 non-null  str  
 5   e                     72606 non-null  int64
 6   n                     72606 non-null  int64
 7   service_code          72606 non-null  str  
 8   service_name          72606 non-null  str  
 9   status                72606 non-null  str  
 10  userid                72606 non-null  int64
 11  title                 72604 non-null  str  
 12  detail                72604 non-null  str  
 13  media_url             49971 non-null  str  
 14  interface_used        72606 non-null  str  
 15  service_notice        71750 non-null  str  
 16  description    

The overwiev and more clear table, shows me that the dataset could use some cleaning. There are some unneccesary columns for my analysis like objectid,agency_sent_datetime, either service_code or service_name, media_url, interface_used_, url. Aditionally, the datetimes are in a sting type, but could be more usful converteded into date types. However before changing the dataset, making a copy is useful, so the original values are not changed.

In the new dataframe "reports_clean" there should be set an index, that is an unique identifier and also clearly states the purpose of the row. In the metadata, the "service_reuquest_id" was stated as a clear service ID in the system. However, first it is important to make sure and ckeck if it is an unique value and therefore can be used as a Index.

In [4]:
reports_clean = reports.copy()
#checking if the service_request_id unique is so it can be used as a index. The index has to be unique, otherwise the reports get mixed up
reports["service_request_id"].is_unique

True

It is unique, so the object_id, which is not stable overtime written in the metadata, can be dropped and the index can be set as service_request_id. The uniqueness of service_request_id also shows that each report appears only once in the dataset, which also checks for duplicate requests. This important because duplicates could influence the analysis negativly.

In [5]:
reports_clean = reports_clean.drop(columns=["objectid"])

reports_clean = reports_clean.set_index("service_request_id")

Now the relevant columns are checked, if they have NaN values These could mess with the the later analsyis.

In [6]:
#requested_datetime is useful to see the date, on which the Report was stated
print(f"It is {reports_clean['requested_datetime'].hasnans} that requested_datetime has NaN")
#requested updated_datetime is useful to see the date, on which a update was made. if the status is fixed, this date represents the date on which the damage was repaired 
print(f"It is {reports_clean['updated_datetime'].hasnans} that updated_datetime has NaN")
#e is useful, because it represents the easting coordinate in a double type
print(f"It is {reports_clean['e'].hasnans} that e (easting) has NaN")
#n is useful, because it represents the northing coordinate in a double type
print(f"It is {reports_clean['n'].hasnans} that n (northing) has NaN")
#service_code is useful, because it represents the category of the issue of the report
print(f"It is {reports_clean['service_code'].hasnans} that service_code has NaN")
#status is useful, because it shows whether the problem has been resolved
print(f"It is {reports_clean['status'].hasnans} that status has NaN")
# detail is useful to get some clear and extensive insight
print(f"It is {reports_clean['detail'].hasnans} that detail has NaN")

It is False that requested_datetime has NaN
It is False that updated_datetime has NaN
It is False that e (easting) has NaN
It is False that n (northing) has NaN
It is False that service_code has NaN
It is False that status has NaN
It is True that detail has NaN


Almost all important columns have n NaN values besides the column detail. However thats not that problematic, because its only more details in text and not the service category which is easier to compare. It is interesting but not that important in the later analysis. Therefore the rows with NaN values are not deleted, but cleaned up by filling it with an empty space.

In [7]:
#now the Nan values are just empty and not missing anymore. 
reports_clean["detail"] = reports_clean["detail"].fillna("")

print(reports_clean["detail"].hasnans)

False


Additionally, the status column was inspected to ensure that the majority of reports are actually marked as fixed. This step is important because the later calculated processing time relies on the assumption that the updated_datetime corresponds to the moment when the issue was resolved. Since this timestamp only reflects the true resolution date when the status is set to fixed, verifying the prevalence of this status is essential to justify the approximation.
Furthermore, the presence of NaN values in the status field was checked to ensure that all reports have a valid status entry. This prevents systematic bias in the processing time calculation and increases the reliability of the approximation.

In [8]:
print(reports_clean["status"].value_counts())
print(f"It is {reports_clean['status'].hasnans} that status has NaN")

status
fixed - council     63469
external             8958
jurisdiction unk       81
not contactable        69
wish                   22
confirmed               7
Name: count, dtype: int64
It is False that status has NaN


Next step is to only keep the useful columns defined before and checked if there are no NaN values. By creating a list with the columns to keep and keep only them.

In [9]:
#create a list with the columns i want to keep in my report instead of dropping the colunns i want to remove makes it less messier
keep_cols = [
    "requested_datetime",
    "updated_datetime",
    "e",
    "n",
    "service_code",
    "status",
    "detail",
]
## Keep only the columns that are relevant for the analysis
reports_clean = reports_clean[keep_cols]
reports_clean.head(3)

,requested_datetime,updated_datetime,e,...,service_code,status,detail
service_request_id,,,,,,,
1,2013-03-14T15:16:15,2013-04-12T07:59:30,2678968,...,Strasse/Trottoir/Platz,fixed - council,Auf dem Asphalt des Bürger...
2,2013-03-14T15:17:57,2013-04-12T08:00:22,2680746,...,Strasse/Trottoir/Platz,fixed - council,Vermessungspunkt ist nicht...
4,2013-03-15T09:14:16,2013-04-12T08:08:10,2684605,...,Strasse/Trottoir/Platz,fixed - council,Beim Trottoir sind einige ...


The last cleaning step ist to convert the dates in date type for later temporal analysis.

In [10]:
# by making a list, i can write a loop, instead of converting both datetime alone.
#a function for only two dates is not necessary as i only need the conversion two times in my project and the loop is short
date_cols = [
    "requested_datetime",
    "updated_datetime"
]

for col in date_cols:
    reports_clean[col] = pd.to_datetime(
        reports_clean[col],
        format="%Y-%m-%dT%H:%M:%S",
    )
    
reports_clean[date_cols].dtypes

requested_datetime    datetime64[us]
updated_datetime      datetime64[us]
dtype: object

Now, there can be calculated new columns based on the converted datetimes. These are very useful for temporal analysis. The year and month are beeing seperated into columns. Additionally alos the prossesing time is beeing calculated and stored into a new variable. The Updated_datetime is mostly the date were a issue has been solved or removed.

In [11]:
reports_clean["year_requested"] = reports_clean["requested_datetime"].dt.year

reports_clean["month_requested"] = reports_clean["requested_datetime"].dt.month

#create a column for the processing time in days, so we ave the direct proccessing time in the column
reports_clean["processing_time_days"] = (
    reports_clean["updated_datetime"] - reports_clean["requested_datetime"]
).dt.total_seconds() / 86400
#check if the columns are in the right dataframe
reports_clean.head(3)

,requested_datetime,updated_datetime,e,...,year_requested,month_requested,processing_time_days
service_request_id,,,,,,,
1,2013-03-14 15:16:15,2013-04-12 07:59:30,2678968,...,2013,3,28.696701
2,2013-03-14 15:17:57,2013-04-12 08:00:22,2680746,...,2013,3,28.696123
4,2013-03-15 09:14:16,2013-04-12 08:08:10,2684605,...,2013,3,27.954097


**saving the Dataframe as a CSV:** After finishing the cleaning and preparing the data frame for the spatial join and later analysis, it's important to safe the dataset back as a csv in the proccesed datafolder.

In [12]:
reports_clean.to_csv("../data/processeddata/zueriwieneu_cleaned.csv", index=True)